# Global Minimum Structure Search for Hf₀.₅Zr₀.₅O₂

This notebook performs an automated search for the lowest-energy configuration of Hf₀.₅Zr₀.₅O₂ by enumerating all possible 50% Hf-to-Zr substitutions and evaluating structures using CHGNet interatomic potentials.

In [ ]:
# Section 1: Import Required Libraries
missing = []
try:
    from itertools import combinations
except ImportError:
    missing.append('itertools')
try:
    from ase.spacegroup import crystal
    from ase.visualize import view
    from ase import io
    from ase import Atoms
    from ase.optimize import BFGS
except ImportError:
    missing.append('ase')
try:
    from chgnet.model import CHGNetCalculator
except ImportError:
    missing.append('chgnet')
try:
    import numpy as np
except ImportError:
    missing.append('numpy')

if missing:
    print(f"Missing required packages: {', '.join(missing)}")
    print("Recommended environment: Python 3.8+ with 'chgnet', 'ase', and 'numpy' installed.")
    print("Install with: pip install chgnet ase numpy")
else:
    print("All libraries imported successfully!")
    print("Recommended environment: Python 3.8+ with 'chgnet', 'ase', and 'numpy' installed.")

In [ ]:
# Section 2: Set User Configuration Parameters
phase = "Pca21"                                    # "Pca21" or "monoclinic"
num_formula_units = 2                             # To reach 24 atoms: 8 Hf + 16 O
relax_atoms = True                                # Whether to relax atoms with CHGNet forces
chgnet_model_path = "my_hf_zr_o_model.pt"       # Update with actual CHGNet model path

output_file = f"Hf0.5Zr0.5O2_global_min_{phase}.cif"

print(f"Configuration:")
print(f"  Phase: {phase}")
print(f"  Formula units: {num_formula_units}")
print(f"  Relax atoms: {relax_atoms}")
print(f"  Output file: {output_file}")

In [ ]:
# Section 3: Generate Initial Crystal Structure
if phase.lower() == "pca21":
    # Pca21 (orthorhombic) example lattice parameters in Å
    a, b, c = 5.1, 5.2, 5.3
    # Hf Wyckoff positions (example)
    hf_positions = [(0, 0, 0), (0.5, 0.5, 0.5)]
    o_positions = [(0.3, 0.3, 0.3), (0.7, 0.7, 0.7),
                   (0.2, 0.5, 0.5), (0.5, 0.2, 0.5)]
    structure = crystal(
        symbols=["Hf"]*len(hf_positions) + ["O"]*len(o_positions),
        basis=hf_positions + o_positions,
        spacegroup=29,  # Pca21
        cellpar=[a, b, c, 90, 90, 90],
        primitive_cell=False
    )
    print(f"Created Pca21 structure with {len(hf_positions)} Hf and {len(o_positions)} O per unit cell")
elif phase.lower() == "monoclinic":
    # Monoclinic P21/c example
    a, b, c, beta = 5.1, 5.2, 5.3, 99.2
    hf_positions = [(0,0,0), (0.5,0.5,0)]
    o_positions = [(0.25,0.25,0.25),(0.75,0.75,0.25),(0.25,0.75,0.75),(0.75,0.25,0.75)]
    structure = crystal(
        symbols=["Hf"]*len(hf_positions) + ["O"]*len(o_positions),
        basis=hf_positions + o_positions,
        spacegroup=14,  # P21/c
        cellpar=[a, b, c, 90, beta, 90],
        primitive_cell=False
    )
    print(f"Created monoclinic structure with {len(hf_positions)} Hf and {len(o_positions)} O per unit cell")
else:
    raise ValueError("Phase must be either 'Pca21' or 'monoclinic'")

# Repeat unit to reach desired number of formula units
structure = structure * (num_formula_units // len(hf_positions))
print(f"Repeated structure to {num_formula_units} formula units")
print(f"Total atoms: {len(structure)}")

In [ ]:
# Section 4: Identify Metal Atom Positions
hf_indices = [i for i, atom in enumerate(structure) if atom.symbol == "Hf"]
num_hf = len(hf_indices)
num_substitute = num_hf // 2  # 50% substitution

print(f"Total Hf atoms: {num_hf}")
print(f"Atoms to substitute (50%): {num_substitute}")
print(f"Total substitution combinations: {np.math.comb(num_hf, num_substitute)}")

In [ ]:
# Section 5: Initialize CHGNet Calculator
try:
    calculator = CHGNetCalculator(model_file=chgnet_model_path)
    structure.set_calculator(calculator)
    print("CHGNet calculator initialized successfully")
except Exception as e:
    print(f"Warning: CHGNet initialization failed: {e}")
    print(f"Please ensure the model path is correct: {chgnet_model_path}")

In [ ]:
# Section 6: Enumerate and Evaluate Substitutions
best_energy = np.inf
best_structure = None
energies_evaluated = []
configs_evaluated = 0

total_configs = np.math.comb(num_hf, num_substitute)
print(f"Starting enumeration of {total_configs} configurations...")
print()

for sub in combinations(hf_indices, num_substitute):
    candidate = structure.copy()
    
    # Perform substitution: Hf -> Zr at selected indices
    for i in sub:
        candidate[i].symbol = "Zr"
    
    # Optional: relax atomic positions while keeping lattice fixed
    if relax_atoms:
        try:
            candidate.set_calculator(calculator)
            dyn = BFGS(candidate, logfile=None)
            dyn.run(fmax=0.01)  # force tolerance in eV/Å
        except Exception as e:
            print(f"Relaxation failed for config {configs_evaluated}: {e}")
            continue
    
    # Energy evaluation
    try:
        energy = candidate.get_potential_energy()
        energies_evaluated.append(energy)
        
        if energy < best_energy:
            best_energy = energy
            best_structure = candidate.copy()
            print(f"New minimum found: Config {configs_evaluated}/{total_configs} - Energy: {energy:.6f} eV")
    except Exception as e:
        print(f"Energy evaluation failed for config {configs_evaluated}: {e}")
    
    configs_evaluated += 1

print()
print(f"Enumeration complete! Evaluated {configs_evaluated} configurations")

In [ ]:
# Section 7: Save Global Minimum Structure
if best_structure is not None:
    print(f"Global minimum energy (Hf0.5Zr0.5O2, {phase}): {best_energy:.6f} eV")
    io.write(output_file, best_structure)
    print(f"Global minimum structure saved to: {output_file}")
    print()
    print("Structure details:")
    print(f"  Cell parameters: {best_structure.cell}")
    print(f"  Number of atoms: {len(best_structure)}")
else:
    print("Error: No valid structures were evaluated. Check CHGNet calculator and model path.")